In [35]:
### Cu 003 processing ###


#%% load the packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib import colors

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment
from defdap.plotting import MapPlot

from pathlib import Path

import copy 
import pandas as pd
import datetime

from scipy.signal import find_peaks
from scipy.interpolate import griddata
from scipy.linalg import lstsq
from scipy.ndimage import median_filter, vectorized_filter, gaussian_filter

from skimage import transform
from skimage.measure import profile_line, block_reduce
from skimage.morphology import binary_dilation, dilation, skeletonize

import os

# get dictools stuff 
import sys
# sys.path.append("c:/work/hrdic-tools/")
# import dictools

plt.rcParams['svg.fonttype'] = 'none'

%matplotlib qt

In [2]:
def lsm_read(lsm_file):
    # for reading in lsm output from ZEISS Confomap

    df = pd.read_csv(lsm_file,names=['x','y','z'])

    x = np.asarray(df['x'])
    y = np.asarray(df['y'])
    z = np.asarray(df['z'])

    # calculate shape - this must be done on the raw data 
    x0 = np.nanmin(x)
    x1 = np.nanmax(x)
    y0 = np.nanmin(y)
    y1 = np.nanmax(y)

    x_size = x1 - x0
    y_size = y1 - y0

    # calculate step size 
    x_step = np.round(np.min(np.abs(np.diff(x))),4)
    y_step = np.round(np.max(np.abs(np.diff(y))),4)


    # create new grid to interpolate data onto
    xg,yg = np.meshgrid(np.arange(x0,x1,x_step),np.arange(y0,y1,y_step))



    # remove the weird way that ConfoMaps saves non-measured points
    x = x[z !='***']
    y = y[z !='***']
    z = z[z !='***']


    # interpolate onto grid to produced gridded data
    zg = griddata(np.asarray([x,y]).T,z,(xg,yg),method='nearest')

    # flip up down for zg
    zg = np.flipud(zg)

    return xg, yg, zg, x_step

def resample_gridded_data(xg,yg,zg,new_step):

    # flatten arrays - we could probably use RegularGridInterpolator but this works for now
    x = xg.flatten()
    y = yg.flatten()
    z = zg.flatten()

    # calculate shape - this must be done on the raw data
    x0 = np.nanmin(x)
    x1 = np.nanmax(x)
    y0 = np.nanmin(y)
    y1 = np.nanmax(y)

    x_size = x1 - x0
    y_size = y1 - y0

    # calculate step size 
    x_step = np.round(np.min(np.abs(np.diff(x))),4)
    y_step = np.round(np.max(np.abs(np.diff(y))),4)

    # create new grid to interpolate data onto
    xg_new,yg_new = np.meshgrid(np.arange(x0,x1,new_step),np.arange(y0,y1,new_step))

    # interpolate onto grid to produced gridded data
    zg_new = griddata(np.asarray([x,y]).T,z,(xg_new,yg_new),method='nearest')

    return xg_new, yg_new, zg_new


def calc_rotations(dic_map):
    # calculate rotations from dic displacement field 

    # extract deformation gradient
    f = dic_map.data.f

    # calculate rotation as ang = (F21 - F12)/2
    rot = (f[0,1,:,:] - f[1,0,:,:])/2

    # centre on mean 
    rot = rot - np.nanmean(rot)

    return rot 

In [3]:

# path to LSM file 
lsm_file = './LSM/raw_surface_v2.txt'

# import data
xg,yg,zg,lsm_step = lsm_read(lsm_file)




# # processing to get data into useful form 
# # crop the rubbish data from edges 
# xL = 800
# xR = 800
# yT = 800
# yB = 800

# # if we want to nanify the deleted data
# # zg[:,:xL] = np.nan
# # zg[:,-xR:] = np.nan

# # zg[:yT,:] = np.nan
# # zg[-yB:,:] = np.nan

# # plt.figure()
# # plt.imshow(zg)

# zg = zg[yT:-yB,xL:-xR]
# xg = xg[yT:-yB,xL:-xR]
# yg = yg[yT:-yB,xL:-xR]
# # plt.figure()
# # plt.imshow(zg)



# best-fit linear plane
A = np.c_[xg.flatten(),yg.flatten(), np.ones(xg.flatten().shape[0])]

C,_,_,_ = lstsq(A, zg.flatten())    # coefficients
    
# fitted plane
zg_fit = C[0]*xg + C[1]*yg + C[2]

# plt.figure()
fig,ax = plt.subplots(1,3)
ax[0].imshow(zg)
ax[0].set_title('Raw surface')
ax[1].imshow(zg_fit)
ax[1].set_title('Fitted flat plane')
ax[2].imshow(zg - zg_fit)
ax[2].set_title('Corrected')

plt.tight_layout()

# corrected surface
zg_flat = zg - zg_fit

# set lowest point on map to zero 
zg_flat = zg_flat - zg_flat.min()

In [4]:
# current dataset is overkill for DIC
# resample to DIC step size 

dic_step_px = 10
dic_px_size = 20/2048
dic_step = dic_step_px*dic_px_size # microns 



xg_new, yg_new, zg_new = resample_gridded_data(xg,yg,zg_flat,dic_step)



In [5]:
# median filter to smooth out bumps from speckle pattern

# # filter kernel size
# k_size = 15

# zg_filtered = median_filter(zg_new,k_size)
zg_filtered = gaussian_filter(zg_new,sigma=5)

fig,ax = plt.subplots()
surf = ax.imshow(zg_filtered,cmap='gist_grey',vmin=0,vmax=2)
bar = plt.colorbar(surf)
bar.set_label('Altitude / μm')

In [6]:
# zg_grad = np.gradient(zg_filtered)
zg_grad = np.gradient(zg_filtered)
zg_grad_mag = ((zg_grad[0]**2 + zg_grad[1]**2)**0.5)/dic_step
# zg_grad_mag = gaussian_filter(zg_grad_mag,sigma=2)

zg_grad_mag = median_filter(zg_grad_mag,10)

fig,ax = plt.subplots()
surf = ax.imshow(zg_grad_mag,cmap='gist_grey',vmin=0.02,vmax=0.2)
bar = plt.colorbar(surf)
bar.set_label('Altitude gradient magnitude / μm/μm')

In [7]:
exp = experiment.Experiment()

# load DIC data 
data_dir = Path('./DIC/pyvale/')
dic_frame = experiment.Frame()

dic_step_list = sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv'))

# for dic_file in sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv')):
#     hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

dic_file = dic_step_list[-2]
hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

hfw = 20.0 # microns
pixelwidth = 2048
pixelsize = hfw/pixelwidth

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.set_scale(pixelsize)
    dic_map.set_crop(left=100,right=100,top=100,bottom=100)
    # dic_map.plot_map('max_shear',vmin=0,vmax=0.01,plot_scale_bar=True)
    print(dic_map)

Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)


In [8]:
# # clip points on DIC map - start bottom left and go ACW

# fig,ax = plt.subplots(1,2)
# surf = ax[0].imshow(zg_grad_mag,cmap='turbo',vmin=0,vmax=0.2)
# bar = plt.colorbar(surf)
# bar.set_label('Altitude gradient magnitude / μm/μm')


# ax[1].imshow(dic_map.data['max_shear'],vmin=0,vmax=0.1)

# dic_click = plt.ginput(5,show_clicks=True,timeout=-1)
# dic_click = np.asarray(dic_click)

# ax[1].plot(dic_click[:,0],dic_click[:,1],'rx')

In [9]:
# # # clip points on zmap map - start bottom left and go ACW

# fig,ax = plt.subplots(1,2)


# ax[1].imshow(dic_map.data['max_shear'],vmin=0,vmax=0.1)
# ax[1].plot(dic_click[:,0],dic_click[:,1],'rx')

# surf = ax[0].imshow(zg_grad_mag,cmap='gist_grey',vmin=0,vmax=0.1)
# bar = plt.colorbar(surf)
# bar.set_label('Altitude gradient magnitude / μm/μm')

# lsm_click = plt.ginput(5,show_clicks=True,timeout=-1)
# lsm_click = np.asarray(lsm_click)

# ax[0].plot(lsm_click[:,0],lsm_click[:,1],'rx')

In [10]:
lsm_click = ([[1927.00974462, 1721.95661122],
       [ 442.76008065, 2944.27986391],
       [2873.97446237, 2272.67368112],
       [3015.01176075,  586.9421623 ],
       [ 530.06888441,  969.75768649]])

dic_click = ([[1578.56451613, 1385.56214718],
       [ 178.35322581, 2692.72142137],
       [2513.51572581, 1957.16738911],
       [2619.8608871 ,  171.45488911],
       [ 196.07741935,  641.14601815]])

In [11]:
tf = transform.ProjectiveTransform()
tf.estimate(lsm_click,dic_click)

zg_grad_mag_warped = transform.warp(zg_grad_mag,tf.inverse,output_shape=dic_map.shape)
zg_filtered_warped = transform.warp(zg_filtered,tf.inverse,output_shape=dic_map.shape)

# we need to shift this to account for the crop in the DIC map
tf_shift = transform.AffineTransform(translation=(100,100))

zg_grad_mag_warped = transform.warp(zg_grad_mag_warped,tf_shift.inverse,output_shape=dic_map.shape)
zg_filtered_warped = transform.warp(zg_filtered_warped,tf_shift.inverse,output_shape=dic_map.shape)


C:\Users\bepoole\AppData\Local\Temp\ipykernel_10728\244900557.py:2: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `ProjectiveTransform.from_estimate` class constructor instead.
  tf.estimate(lsm_click,dic_click)


In [12]:
# surely it can't be this simple! 
# let's try to bodge it in to defdap 
dic_map.data.add(
    'lsm_height', zg_filtered_warped,
    unit='μm', type='map', order=0,
    plot_params={
        'plot_colour_bar': True,
        'clabel': 'Height',
    }
)

dic_map.data.add(
    'lsm_height_grad', zg_grad_mag_warped,
    unit='μm/μm', type='map', order=0,
    plot_params={
        'plot_colour_bar': True,
        'clabel': 'Height gradient',
    }
)


In [13]:

# link ebsd  map
ebsd_frame = experiment.Frame()
data_dir = Path('.')
ebsd.Map(data_dir / 'Pre_EBSD/map.cpr',
         increment=exp.increments[0], frame=ebsd_frame)

ebsd_map = exp.increments[0].maps['ebsd']
# ebsd_map.set_homog_point()

dic_map = exp.increments[0].maps['hrdic']

# dic_map.set_homog_point(vmin=0,vmax=0.05)

ebsd_frame.homog_points = [(1946, 1565),
 (2443, 1000),
 (1305, 1027),
 (1395, 2225),
 (2572, 2193),
 (1876, 1077),
 (2613, 1497),
 (1822, 2208),
 (1259, 1661),
 (2229, 1260),
 (1641, 1325),
 (1693, 1794),
 (2195, 1762)]

dic_frame.homog_points = [(1582, 1380),
 (2615, 172),
 (238, 226),
 (453, 2748),
 (2882, 2728),
 (1431, 326),
 (2970, 1240),
 (1329, 2732),
 (157, 1568),
 (2170, 731),
 (941, 863),
 (1061, 1854),
 (2100, 1795)]

ebsd_map = exp.increments[0].maps['ebsd']

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.link_ebsd_map(ebsd_map, transform_type="polynomial",order=2)
    # dic_map.link_ebsd_map(ebsd_map, transform_type="affine")

Loaded EBSD data (dimensions: 3727 x 2795 pixels, step size: 0.2 um)


In [14]:
fig,ax = plt.subplots(1,2)

dic_map.plot_map('max_shear', plot_gbs='line',plot_scale_bar=True,boundary_colour='white',cmap='viridis',vmin=0,vmax=0.1,fig=fig,ax=ax[0])
dic_map.plot_map('lsm_height_grad',plot_gbs='line',plot_scale_bar=True,boundary_colour='white',cmap='afmhot',vmin=0,vmax=0.1,fig=fig,ax=ax[1])

plt.tight_layout()

Finished building quaternion array (0:00:25) 
Finished finding grain boundaries (0:01:14) 


C:\Ben\Work\DefDAP\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(
C:\Ben\Work\DefDAP\defdap\experiment.py:138: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mph.remove_small_objects(points_img, min_size=10, connectivity=2,


In [15]:
dic_map.plot_map('lsm_height',plot_gbs='line',plot_scale_bar=True,boundary_colour='white',cmap='afmhot',vmin=0,vmax=3)

In [16]:
def height_map_line_profiler(dic_map,click_on=True,line_profile_width=3,points=" "):
    # line profiles arrays of interest 
    ms_map = copy.deepcopy(dic_map.data['max_shear'])
    # ms_map = vectorized_filter(dic_map.data['max_shear'],function=np.nanmedian,size=3)
    ms_map[np.isnan(ms_map)] = 0.1
    # ms_map = median_filter(dic_map.data['max_shear'],31)

    # height maps
    height_map = dic_map.data['lsm_height']
    height_grad_map = dic_map.data['lsm_height_grad']

    # gb map
    gb_map = dic_map.data['grain_boundaries'].image

    # dilate gb_map 
    gb_map = binary_dilation(gb_map,np.ones((2,2)))
    # plt.figure()
    # plt.imshow(gb_map)


    # extract some line profiles 
    if click_on == True:

        plt.figure()
        # plt.imshow(dic_map.data['lsm_height2'],alpha=1,cmap='Blues')
        plt.imshow(ms_map,alpha=1,cmap='viridis',vmin=0,vmax=0.1)

        plt.xlim([0,2997])
        plt.ylim([2997,0])

        pts = plt.ginput(2,show_clicks=True,timeout=-1)

        pts = np.array(pts)

    else: 
        pts = points

    x0 = pts[0,0]
    x1 = pts[1,0]

    y0 = pts[0,1]
    y1 = pts[1,1]

    profile_length = dic_map.scale*((x1 - x0)**2 + (y1 - y0)**2)**0.5

    plt.figure()
    plt.imshow(ms_map,alpha=1,cmap='viridis',vmin=0,vmax=0.1)

    plt.plot((x0,x1),(y0,y1),'k-')
    plt.plot(x0,y0,'x',markersize=10)
    plt.plot(x1,y1,'o',markersize=10)



    #interpolate over datasets we want to get line profiles from 
    ms_profile = profile_line(ms_map,(y0,x0),(y1,x1),linewidth=line_profile_width)
    height_profile = profile_line(height_map,(y0,x0),(y1,x1),linewidth=line_profile_width)
    height_profile = height_profile - height_profile.min()

    height_grad_profile = profile_line(height_grad_map,(y0,x0),(y1,x1),linewidth=line_profile_width)

    gb_profile = profile_line(gb_map,(y0,x0),(y1,x1),linewidth=1)

    prof_pos = np.linspace(0,profile_length,len(ms_profile))

    gb_peaks = find_peaks(gb_profile)

        
    # do the plot 
    fig = plt.figure()

    fig.set_size_inches(10,3)

    gs = GridSpec(3,3,width_ratios=[4,5,5])
    ax0 = fig.add_subplot(gs[0,:1])
    ax1 = fig.add_subplot(gs[1,:1])
    ax2 = fig.add_subplot(gs[2,:1])
    ax3 = fig.add_subplot(gs[:,1:2])
    ax4 = fig.add_subplot(gs[:,2:])

    ax0.plot(prof_pos, ms_profile,'-',markeredgecolor='teal',color='teal')
    ax0.set_ylabel(r'$ϵ_{\mathrm{eff}} / -$')


    ax1.plot(prof_pos, height_profile,'-',markeredgecolor='orange',color='orange')
    ax1.set_ylabel(r'$z$ / μm')


    ax2.plot(prof_pos, height_grad_profile,'-',markeredgecolor='darkorange',color='darkorange')
    ax2.set_ylabel(r'$|\nabla(z)|$ / μm/μm')


    # gb zone 
    gb_zone_width = 2 # microns 
    gbx0 = prof_pos[gb_peaks[0]] - gb_zone_width/2
    gbx1 = prof_pos[gb_peaks[0]] + gb_zone_width/2

    for ax in [ax0,ax1,ax2]:
        for i in range(len(gbx0)):
            ax.axvspan(gbx0[i],gbx1[i],alpha=0.2,color='purple')
        ax.grid()
        ax.set_ylim(bottom=0)

    ax2.set_xlabel('Position along path / μm')

    #change zero point of height map 

    # dic_map.plot_map('max_shear', 
    #             plot_gbs='pixel',
    #             dilate_boundaries = True,
    #             plot_scale_bar=True,
    #             boundary_colour='white',
    #             cmap='viridis',
    #             vmin=0,vmax=0.1,
    #             fig=fig,
    #             ax=ax3)
    
    # using out own max shear map with nans corrected
    MapPlot.create(dic_map,
                   ms_map,
                   plot_gbs='line',
                   plot_scale_bar=True,
                   boundary_colour=(1,1,1,0.5),
                   plot_colour_bar=True,
                   clabel='Effective strain / -',
                   cmap='viridis',
                   vmin=0,vmax=0.1,
                   fig=fig,
                   ax=ax3)
    
    dic_map.plot_map('lsm_height_grad', 
                     plot_gbs='line',
                     plot_scale_bar=True,
                     boundary_colour=(1,1,1,0.5),
                     cmap='afmhot',
                     vmin=0,vmax=0.2,
                     fig=fig,
                     ax=ax4
                     )
    
    for ax in [ax3,ax4]:

        ax.plot((x0,x1),(y0,y1),'w-',linewidth=2)
        ax.plot(x0,y0,'x',markeredgecolor='w',markersize=10)
        ax.plot(x1,y1,'o',markeredgecolor='w',markerfacecolor='k',markersize=10)
        
    #     # if x1 > x0: 
    #     #     ax.set_xlim([ x1 - 100, x0 + 100])
    #     # else: 
    #     #     ax.set_xlim([ x0+100, x1 - 100])

    #     # if y1 > y0: 
    #     #     ax.set_ylim([ y1 - 100, y0 + 100])
    #     # else: 
    #     #     ax.set_ylim([ y0+100, y1 - 100])

        # plotting centre 
        xc = np.mean([x0,x1])
        yc = np.mean([y0,y1])

        # make it so it's square 
        xdiff = abs(xc - x0)
        ydiff = abs(yc - y0)

        # find max 
        box_half_width = np.max([xdiff,ydiff])+0.2*profile_length/dic_map.scale

        
        ax.set_xlim([xc - box_half_width, xc + box_half_width])
        ax.set_ylim([yc + box_half_width, yc - box_half_width])



    plt.tight_layout()

    return pts


In [ ]:
# fig,ax = plt.subplots()

# dic_map.plot_map('max_shear',
#                  vmin=0,
#                  vmax=0.1,
#                  plot_gbs='pixel',
#                  dilate_boundaries=True,
#                  plot_scale_bar=True,
#                  fig=fig,
#                  ax=ax)

# # out of plane points 1 
# colour = 'hotpink'
# pts = np.array([[2009.35227273, 1251.99350649],
#        [2106.65746753, 1130.36201299]])

# ax.plot(pts[:,0],pts[:,1],
#         linewidth=3,
#         color=colour)

# ax.plot(pts[0,0],pts[0,1],
#         marker='x',
#         markeredgecolor=colour,
#         markersize=10,
#         markeredgewidth=3)

# ax.plot(pts[1,0],pts[1,1],
#         marker='o',
#         markeredgecolor=colour,
#         markerfacecolor='white',
#         markersize=10,
#         markeredgewidth=3)



# # out of plane points 2
# colour = 'hotpink'
# pts = np.array([[1522.8262987 , 1333.08116883],
#        [1644.45779221, 1333.08116883]])

# ax.plot(pts[:,0],pts[:,1],
#         linewidth=3,
#         color=colour)

# ax.plot(pts[0,0],pts[0,1],
#         marker='x',
#         markeredgecolor=colour,
#         markersize=10,
#         markeredgewidth=3)

# ax.plot(pts[1,0],pts[1,1],
#         marker='o',
#         markeredgecolor=colour,
#         markerfacecolor='white',
#         markersize=10,
#         markeredgewidth=3)


# # in plane points 1
# colour = 'lightsalmon'
# pts = np.array([[1490.51598402,   79.84015984],
#        [1578.34015984,  107.78421578]])

# ax.plot(pts[:,0],pts[:,1],
#         linewidth=3,
#         color=colour)

# ax.plot(pts[0,0],pts[0,1],
#         marker='x',
#         markeredgecolor=colour,
#         markersize=10,
#         markeredgewidth=3)

# ax.plot(pts[1,0],pts[1,1],
#         marker='o',
#         markeredgecolor=colour,
#         markerfacecolor='white',
#         markersize=10,
#         markeredgewidth=3)

# # in plane points 2
# colour = 'lightsalmon'
# pts = np.array([[2041.41308691,  598.8011988 ],
#        [2189.11738262,  307.38461538]])

# ax.plot(pts[:,0],pts[:,1],
#         linewidth=3,
#         color=colour)

# ax.plot(pts[0,0],pts[0,1],
#         marker='x',
#         markeredgecolor=colour,
#         markersize=10,
#         markeredgewidth=3)

# ax.plot(pts[1,0],pts[1,1],
#         marker='o',
#         markeredgecolor=colour,
#         markerfacecolor='white',
#         markersize=10,
#         markeredgewidth=3)

# ax.set_xlim([1250,2450])
# ax.set_ylim([1500,20])

# # plt.savefig('figures_for_paper/Figure_5/line_profile_positions.png',dpi=1000)

C:\Ben\Work\DefDAP\defdap\plotting.py:478: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  boundaries_image = mph.binary_dilation(boundaries_image)


(1500.0, 20.0)

In [203]:
# mixed sliding example

pts = np.array([[1522.8262987 , 1333.08116883],
       [1644.45779221, 1333.08116883]])


height_map_line_profiler(dic_map,click_on=False,line_profile_width=3,points=pts )
plt.savefig('figures_for_paper/Figure_5/mixed_line.svg',dpi=300)

C:\Users\bepoole\AppData\Local\Temp\ipykernel_10728\2260575085.py:16: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  gb_map = binary_dilation(gb_map,np.ones((2,2)))


In [204]:
# in plane sliding
pts = np.array([[1937.62087912, 1465.06693307],
       [2045.40509491, 1604.78721279]])

height_map_line_profiler(dic_map,click_on=False,line_profile_width=3,points=pts )
plt.savefig('figures_for_paper/Figure_5/ip_line.svg',dpi=300)

C:\Users\bepoole\AppData\Local\Temp\ipykernel_10728\2260575085.py:16: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  gb_map = binary_dilation(gb_map,np.ones((2,2)))


In [205]:
# out-of-plane sliding
pts = np.array([[352.79370629, 562.87312687],
       [380.73776224, 654.68931069]])

height_map_line_profiler(dic_map,click_on=False,line_profile_width=3,points=pts)

plt.savefig('figures_for_paper/Figure_5/oop_line.svg',dpi=300)

C:\Users\bepoole\AppData\Local\Temp\ipykernel_10728\2260575085.py:16: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  gb_map = binary_dilation(gb_map,np.ones((2,2)))


Un-use additional work

In [199]:
factor = 10
percentile = 100

# tease out GBs
gbs = dic_map.data['grain_boundaries'].image
# gbs = binary_dilation(gbs,np.ones((5,5)))


# medianify 
gbs_binned = block_reduce(gbs[:-100,:-100],factor,func=np.nanmax)
gbs_binned = binary_dilation(gbs_binned,np.ones((2,2)))

ms_binned = block_reduce(dic_map.data['max_shear'][:-100,:-100],factor*2,func=np.nanpercentile,func_kwargs={'q':percentile})
hgrad_binned = block_reduce(dic_map.data['lsm_height_grad'],factor*2,func=np.nanpercentile,func_kwargs={'q':percentile})

ms_binned = transform.resize(ms_binned,gbs_binned.shape)
hgrad_binned = transform.resize(hgrad_binned,gbs_binned.shape)


# fig,ax = plt.subplots(1,2)
# ax[0].imshow(ms_binned,vmin=0,vmax=0.1,cmap='viridis')
# ax[1].imshow(hgrad_binned,vmin=0,vmax=0.2,cmap='afmhot')

# plt.figure()
# plt.imshow(gbs_binned)

# plt.figure()
# plt.imshow(ms_binned,vmin=0,vmax=0.1,cmap='viridis')

# plt.figure()
# plt.imshow(hgrad_binned,vmin=0,vmax=0.2,cmap='afmhot')

# fig,ax = plt.subplots()
# ax.plot(ms_binned[gbs_binned],hgrad_binned[gbs_binned],'+',alpha=0.1)
# # ax.set_xlim([0,0.2])
# ax.set_xscale('log')
# # ax.set_ylim([0,0.2])
# ax.set_yscale('log')
# xline = np.logspace(-1.5,1.5,10)


# ax.plot(xline,xline)

ms_binned[~gbs_binned] = np.nan
hgrad_binned[~gbs_binned] = np.nan

plt.figure()
plt.imshow(ms_binned,vmin=0,vmax=0.2)
plt.title('max shear')
plt.figure()
plt.imshow(hgrad_binned,vmin=0,vmax=0.1)
plt.title('zgrad')

inplane_yes = ms_binned>np.nanpercentile(ms_binned,75)
outplane_yes = hgrad_binned>np.nanpercentile(hgrad_binned,75)

slidosity = 2*gbs_binned + 2*inplane_yes.astype(int) + 4*outplane_yes.astype(int)

# gbs     in      out
# 0        0        0   = 0
# 2        0        0   = 2
# 2        2        0   = 4
# 2        2        4   = 8
# 2        0        4   = 6
        

# slidosity = dilation(slidosity,np.ones((4,4)))
slidosity_mask = skeletonize(slidosity)
slidosity_mask = slidosity_mask.astype(bool)

slidosity[~slidosity_mask] = 0



# reshape to normal DIC map for platting 
# slidosity = transform.resize(slidosity,(dic_map.shape[0]-101,dic_map.shape[1]-101),preserve_range=True)

slidosity = dilation(slidosity,np.ones((2,2)))
# slidosity = median_filter(slidosity,4)

five_colours = colors.ListedColormap(['white','gainsboro','lightsalmon','hotpink','lawngreen'])
bounds = [-1,1,3,5,7,9]
norm = colors.BoundaryNorm(bounds,five_colours.N)
fig,ax = plt.subplots()
fig.set_size_inches(6,4)

# MapPlot.create(dic_map,
#                 slidosity,
#                 plot_gbs='line',
#                 plot_scale_bar=True,
#                 boundary_colour=(1,1,1,0.2),
#                 # plot_colour_bar=True,
#                 # clabel='Effective strain / -',
#                 cmap=three_colours,
#                 fig=fig,
#                 ax=ax)


im = ax.imshow(slidosity,cmap=five_colours,norm=norm,alpha=1)

ax.axes.get_xaxis().set_ticks([])
ax.axes.get_yaxis().set_ticks([])
cbar = fig.colorbar(im,ticks=[0,2,4,6,8],orientation='vertical')
cbar.ax.set_yticklabels(['cores','Minimal activity','in-plane dominated','out-of-plane dominated','Mixed'])

plt.tight_layout()
plt.savefig('figures_for_paper/Figure_5/slidosity.svg',dpi=1000)







C:\Ben\Work\DefDAP\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(
C:\Ben\Work\DefDAP\defdap\experiment.py:138: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mph.remove_small_objects(points_img, min_size=10, connectivity=2,
C:\Users\bepoole\AppData\Local\Temp\ipykernel_10728\2099443072.py:11: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric foot

In [84]:
non_sliding_boundaries.shape

(580, 580)

In [50]:
dic_map.data['max_shear'].shape

(2997, 2997)

In [ ]:


fig,ax = plt.subplots()
plt.hist2d(ms_binned[gbs_binned]/np.nanmean(ms_binned[gbs_binned]),hgrad_binned[gbs_binned]/np.nanmean(hgrad_binned[gbs_binned]),bins=np.logspace(-1,1,50),norm=colors.LogNorm(vmin=1e-3,vmax=1e2),density=True)
plt.xscale('log')
plt.yscale('log')
plt.colorbar()
plt.xlim([10**-1,10**1])
plt.ylim([10**-1,10**1])


plt.figure()
plt.hist(ms_binned[gbs_binned],bins=100,range=(0,0.2))

plt.figure()
plt.hist(hgrad_binned[gbs_binned],bins=100,range=(0,0.2))